[Cell 1]
# LangChain으로 RAG 시작하기 (OpenAI 버전)

**수정**: OpenAI API 사용 버전 (PyCharm 로컬 실행)

---
RAG(Retrieval-Augmented Generation) 파이프라인을
**OpenAI API + PyCharm 로컬 환경**에서 직접 구현

### 실습 순서
1. **환경 설정** — 패키지 설치 & API 키 등록
2. **LLM 사용해보기** — OpenAI 모델 호출
3. **Schema 사용해보기** — 메시지 역할 구분
4. **Prompt Template** — 프롬프트 엔지니어링
5. **Parser** — 출력 파싱
6. **Chain** — 파이프라인 연결
7. **Document Loader** — PDF/CSV/Web 문서 로드
8. **Text Splitter** — 텍스트 분할 (Recursive vs Character)
9. **Embedding** — 텍스트 → 벡터 변환
10. **VectorStore** — 벡터 저장소 구축
11. **Retriever + QA** — RAG 완성!


[Cell 2]
## Step 0: 환경 설정


```bash


#  패키지 설치
pip install langchain langchain-openai langchain-community
pip install chromadb pypdf tiktoken
pip install jupyter ipykernel
pip install numpy pandas beautifulsoup4 python-dotenv


# 실습용 PDF 다운로드 (Demian.pdf)
 -> 실습 파일  로칼에서 직접 사용
```



[Cell 3] ### API 키 등록 및 사용

`.env` 파일에 저장된 OpenAI API 키를 로드 하는 은닉 방식 사용.



In [7]:
# [Cell 4]
import os
from dotenv import load_dotenv

# .env 파일로부터 환경변수 로드
load_dotenv()


True

[Cell 5]
#### Step 1: LLM 사용해보기

original code에 사용된 `ChatGoogleGenerativeAI`대신   `ChatOpenAI`를 사용


In [8]:
# [Cell 6]
from langchain_openai import ChatOpenAI

# LLM 선언 (ChatOpenAI 사용)
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.0)

# invoke()로 질문하기
# invoke는 LangChain에서 LLM이나 Chain을 실행할 때 사용하는 메서드
request = llm.invoke("파이썬에 대해 간단히 설명해줘")
print(request)


content='파이썬은 간결하고 읽기 쉬운 문법을 가진 프로그래밍 언어로, 다양한 용도로 사용되는 인기 있는 언어입니다. 파이썬은 다양한 라이브러리와 모듈을 제공하여 데이터 분석, 인공지능, 웹 개발 등 다양한 분야에서 활용됩니다. 또한, 파이썬은 무료이며 오픈 소스이기 때문에 누구나 쉽게 배우고 사용할 수 있습니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 157, 'prompt_tokens': 28, 'total_tokens': 185, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DKCbu00ARmELkmMOnK72lkoARXBVh', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='run--019cf933-2a50-7673-b450-f20cc0df22ed-0' usage_metadata={'input_tokens': 28, 'output_tokens': 157, 'total_tokens': 185, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


[Cell 7]
### content만 출력하기

응답 객체에는 답변 뿐만 아니라 메타데이터도 포함되어 있습니다.  
실제 답변 텍스트만 보려면 `.content`를 사용합니다.


In [9]:
# [Cell 8]
# 답변 텍스트만 출력
print(request.content)


파이썬은 간결하고 읽기 쉬운 문법을 가진 프로그래밍 언어로, 다양한 용도로 사용되는 인기 있는 언어입니다. 파이썬은 다양한 라이브러리와 모듈을 제공하여 데이터 분석, 인공지능, 웹 개발 등 다양한 분야에서 활용됩니다. 또한, 파이썬은 무료이며 오픈 소스이기 때문에 누구나 쉽게 배우고 사용할 수 있습니다.


[Cell 9]
## Step 2: Schema

메시지를 역할별로 구분하여 대화 흐름을 관리합니다.  
앞서 [Cell 6]에서 생성한 `llm` 객체를 그대로 사용합니다.

- **SystemMessage**: LLM에 역할/제약 부여 (예: "너는 번역가야")
- **HumanMessage**: 사용자가 보내는 메시지
- **AIMessage**: AI의 응답



In [10]:
# [Cell 10]
from langchain.schema import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage(content="You are a helpful assistant that translates English to Korean."), #서버 개념
    HumanMessage(content="Translate this sentence: I love programming.") #클라이언트 개념
]

# OpenAI는 SystemMessage를 기본 지원!
# [Cell 6]에서 정의한 llm 객체를 사용하여 호출합니다.
response = llm.invoke(messages)
print(response.content)


나는 프로그래밍을 사랑해요.


In [11]:
# [Cell 11]
# 응답 객체 전체 확인 — AIMessage로 분류되어 있음
response


AIMessage(content='나는 프로그래밍을 사랑해요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 30, 'total_tokens': 44, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DKCbw4tCWLOLIbpm7EwtOSQAjKCyW', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019cf933-329e-7b10-b5ba-e2694d55977d-0', usage_metadata={'input_tokens': 30, 'output_tokens': 14, 'total_tokens': 44, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

[Cell 12]
## Step 3: Prompt Template 사용해보기

반복되는 프롬프트 구조를 템플릿으로 만들어 재사용합니다.  
`{변수명}` 형태로 동적 부분을 지정할 수 있어요.

**예시:**
- "파란색 셔츠"와 잘 어울리는 "바지"를 추천해줘!
- "흰 티셔츠"와 잘 어울리는 "모자"를 추천해줘!

→ `"{상의}"와 잘 어울리는 "{하의}"를 추천해줘!` 로 템플릿화!


In [12]:
# [Cell 13]
from langchain.prompts import PromptTemplate

# PromptTemplate: 일반적인 프롬프트 템플릿
prompt = PromptTemplate.from_template("{meat}를 맛있게 만드는 좋은 방법은?")

# format()으로 변수에 값을 넣어 완성
formatted = prompt.format(meat="닭고기")#format 메서드 :{meat} 자리에 "닭고기"를 넣어줘
print(formatted)


닭고기를 맛있게 만드는 좋은 방법은?


[Cell 14]
### ChatPromptTemplate

채팅 LLM에 특화된 템플릿으로, 역할(system/human)을 나누어 구성합니다.


In [13]:
# [Cell 15]
from langchain.prompts import ChatPromptTemplate
template = "당신은 최고의 번역가입니다. {input_language}를 {output_language}로 번역해주세요."
human_template = "{text}"

chat_prompt = ChatPromptTemplate.from_messages([ #블록 조립
    ("system", template),
    ("human", human_template),
])

# 변수에 값을 넣어 메시지 리스트 생성
format_message = chat_prompt.format_messages( #블록 조립
    input_language="English",
    output_language="Korean",
    text="I really love pizza."
)

print(format_message)#이는 AI 에게 보내는 메세지 임
'''
format_messages가 하는 구체적인 일
1.빈칸 채우기 (치환): 미리 준비된 대본 속의 {input_language}, {output_language}, {text}라는 3개의 빈칸에 각각 "English", "Korean", **"I really love pizza."**라는 실제 내용을 채워 넣기
2.봉투에 담기 (객체화): 빈칸이 채워진 문장들을 그냥 두지 않고, AI가 읽기 편하도록 **'시스템용 봉투'**와 **'사용자용 봉투'**에 나누어 담습니다.
3.꾸러미 만들기 (리스트화): 이 봉투들을 순서대로 모아서 AI에게 한꺼번에 전달할 수 있는 '하나의 메시지 꾸러미'(format_message)로 완성합니다.
비유: 미리 뽑아둔 **'설문지 양식'**에 이름, 날짜, 내용을 실제로 적어 넣어서 제출할 수 있는 **'완성된 서류'**로 만드는 과정입니다!
이 과정을 거쳐야 비로소 AI가 읽고 답장을 줄 수 있는 상태가 됩니다.

'''

[SystemMessage(content='당신은 최고의 번역가입니다. English를 Korean로 번역해주세요.', additional_kwargs={}, response_metadata={}), HumanMessage(content='I really love pizza.', additional_kwargs={}, response_metadata={})]


'\nformat_messages가 하는 구체적인 일\n1.빈칸 채우기 (치환): 미리 준비된 대본 속의 {input_language}, {output_language}, {text}라는 3개의 빈칸에 각각 "English", "Korean", **"I really love pizza."**라는 실제 내용을 채워 넣기\n2.봉투에 담기 (객체화): 빈칸이 채워진 문장들을 그냥 두지 않고, AI가 읽기 편하도록 **\'시스템용 봉투\'**와 **\'사용자용 봉투\'**에 나누어 담습니다.\n3.꾸러미 만들기 (리스트화): 이 봉투들을 순서대로 모아서 AI에게 한꺼번에 전달할 수 있는 \'하나의 메시지 꾸러미\'(format_message)로 완성합니다.\n비유: 미리 뽑아둔 **\'설문지 양식\'**에 이름, 날짜, 내용을 실제로 적어 넣어서 제출할 수 있는 **\'완성된 서류\'**로 만드는 과정입니다!\n이 과정을 거쳐야 비로소 AI가 읽고 답장을 줄 수 있는 상태가 됩니다.\n\n'

In [14]:
# [Cell 16]
# 포맷된 메시지를 LLM에 전달
response = llm.invoke(format_message)#invoke 부르다 작동시키다
print(response.content)


나는 피자를 정말 좋아해.


[Cell 17]
## Step 4: Parser 사용해보기

Parser는 LLM 응답 텍스트를 원하는 형태로 변환(파싱)합니다.  
예: 쉼표로 구분된 문자열 → Python 리스트


In [15]:
# [Cell 18]
from langchain.schema import BaseOutputParser

class CommaSeparatedListOutputParser(BaseOutputParser):
    """쉼표를 기준으로 문자열을 리스트로 분리하는 Parser"""
    def parse(self, text: str) -> list: #프로그램이 작업하기 용이하게 리스크(목록) 형태로 변환
        return text.strip().split(", ")

# 테스트
CommaSeparatedListOutputParser().parse("사과, 바나나, 포도")


['사과', '바나나', '포도']

[Cell 19]
## Step 5: Chain 사용하여 결합해보기

Chain은 LangChain의 **핵심 기능**입니다!  
`|` (파이프) 기호로 Prompt → LLM → Parser를 한 번에 연결합니다.

```
chat_prompt | llm | parser
```

이것이 바로 수업에서 배운 **"Prompt → Model → Output Parser"** 흐름이에요.


In [16]:
# [Cell 20]
template = """당신은 쉼표로 구분된 목록을 생성하는 유용한 조수입니다.
사용자가 카테고리를 전달하면 해당 카테고리에 속하는 5개의 객체를 쉼표로 구분된 목록으로 생성합니다.
오직 쉼표로 구분된 목록만 반환하고 그 이상은 반환하지 마세요."""

human_template = "{text}"

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", template),
    ("human", human_template),
])

# Prompt + LLM + Parser 를 Chain으로 연결!
chain = chat_prompt | llm | CommaSeparatedListOutputParser()#깨끗하게 리스트 형태로 정리

# 실행
response = chain.invoke({"text": "와인의 종류"})
print(response)


['피노 누아르', '시라', '샤르도네', '메를로', '까베르네 소비뇽']


In [17]:
# [Cell 21]
# 다른 카테고리로도 시도해보세요!
response2 = chain.invoke({"text": "조선의 왕들"})
print(response2)


['세종대왕', '태조이성계', '성종대왕', '고종대왕', '명종대왕']


[Cell 22]


#### 여기서부터 RAG 파이프라인 구축!

위에서 학습한 LLM, Prompt, Chain을 활용해서
**문서 기반 질의응답(RAG)** 시스템을 만들기.

---


[Cell 23]
#### Step 6: Document Loader 사용해보기

Document Loader는 다양한 형태의 원본 데이터를 LLM이 이해할 수 있는 **Document 객체(text + metadata)** 로 변환하는 역할을 합니다.

PDF, 웹페이지, CSV와 같이 형식이 서로 다른 문서들을 일관된 구조로 파싱하여, 이후 Chunking·Embedding·검색(Retrieval) 단계에서 바로 사용할 수 있도록 만들어줍니다.

즉, Document Loader는 RAG 파이프라인의 가장 첫 단계에서 **“데이터를 읽을 수 있는 형태로 정리"**하는 역할을 담당합니다.

> 🔗 [공식 문서: Document Loaders 목록](https://python.langchain.com/docs/modules/data_connection/document_loaders/)

---

### 1) PDFLoader 사용
가장 많이 사용되는 문서 형식인 PDF 파일을 대상으로 `PyPDFLoader`를 사용해 문서를 불러옵니다.
PDFLoader는 각 페이지를 하나의 Document 단위로 변환하며, 이후 Text Splitter를 통해 의미 단위로 다시 분할됩니다.

> `Demian.pdf` 파일이 프로젝트 로칼 디렉토리에 위치 (현재 182페이지 분량)


In [18]:
# [Cell 24]
from langchain_community.document_loaders import PyPDFLoader

# PDF 파일 로드
loader = PyPDFLoader("Demian.pdf")
pages = loader.load_and_split()

# 총 페이지 수 확인
print(f"총 {len(pages)}개 페이지 로드됨")


총 182개 페이지 로드됨


In [19]:
# [Cell 25]
# 첫 번째 페이지의 Document 객체 확인
print(" 첫 번째 페이지 미리보기:")
print(pages[0])


 첫 번째 페이지 미리보기:
page_content='DEMIAN 
• 
Downloaded from https://www.holybooks.com' metadata={'producer': 'Adobe Acrobat Standard DC 19 Paper Capture Plug-in', 'creator': 'ScanFix(TM) Enhanced', 'creationdate': '2015-09-10T01:40:29+00:00', 'moddate': '2019-01-30T17:47:47+01:00', 'source': 'Demian.pdf', 'total_pages': 182, 'page': 0, 'page_label': '1'}


In [20]:
# [Cell 26]
# 실제 텍스트 본문(page_content)만 선택하여 확인
print(f"11번째 페이지 본문 미리보기:")
print("-" * 30)
print(pages[10].page_content)


11번째 페이지 본문 미리보기:
------------------------------
TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smalle

[Cell 27]
### 2) CSVLoader 사용
CSV 파일은 행(row) 단위로 구조화된 데이터를 담고 있는 형식으로, `CSVLoader`를 사용하면 각 행을 하나의 Document 객체로 변환할 수 있습니다.


In [21]:
# [Cell 28]
import pandas as pd
# 실습을 위해 타이타닉 데이터셋 다운로드
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
pd.read_csv(url).to_csv("titanic.csv", index=False)
print(" titanic.csv 다운로드 완료!")


 titanic.csv 다운로드 완료!


In [22]:
# [Cell 29]
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader("titanic.csv")
data = loader.load()

# 상위 3개 데이터 확인
data[:3]


[Document(metadata={'source': 'titanic.csv', 'row': 0}, page_content='PassengerId: 1\nSurvived: 0\nPclass: 3\nName: Braund, Mr. Owen Harris\nSex: male\nAge: 22.0\nSibSp: 1\nParch: 0\nTicket: A/5 21171\nFare: 7.25\nCabin: \nEmbarked: S'),
 Document(metadata={'source': 'titanic.csv', 'row': 1}, page_content='PassengerId: 2\nSurvived: 1\nPclass: 1\nName: Cumings, Mrs. John Bradley (Florence Briggs Thayer)\nSex: female\nAge: 38.0\nSibSp: 1\nParch: 0\nTicket: PC 17599\nFare: 71.2833\nCabin: C85\nEmbarked: C'),
 Document(metadata={'source': 'titanic.csv', 'row': 2}, page_content='PassengerId: 3\nSurvived: 1\nPclass: 3\nName: Heikkinen, Miss. Laina\nSex: female\nAge: 26.0\nSibSp: 0\nParch: 0\nTicket: STON/O2. 3101282\nFare: 7.925\nCabin: \nEmbarked: S')]

[Cell 30]
### 3) WebBaseLoader 사용
웹페이지의 텍스트 콘텐츠를 직접 파싱하여 Document 객체로 변환합니다. 뉴스 기사나 블로그 글 등을 실시간 지식 소스로 활용할 수 있습니다.


In [36]:
# [Cell 31]
import os
from langchain_community.document_loaders import WebBaseLoader

# WebBaseLoader 사용 시 User-Agent를 설정하여 경고를 방지합니다.
os.environ["USER_AGENT"] = "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"

# 뉴스 기사 로드 예시
loader = WebBaseLoader("https://it.chosun.com/news/articleView.html?idxno=2023092111831")
documents = loader.load()

print(f" 웹페이지 본문 길이: {len(documents[0].page_content)}자")
# print(documents[0].page_content) # 주석을 해제하면 본문 전체 확인 가능


 웹페이지 본문 길이: 5130자


[Cell 32]


## Step 7: Text Splitter 사용해보기

긴 문서를 작은 **Chunk(덩어리)** 로 분할합니다. LLM의 토큰 제한을 극복하고 검색 효율을 높이기 위한 **필수 단계**입니다.

### 1) Splitter의 종류
*   **CharacterTextSplitter**: 하나의 고정된 구분자(예: `\n\n`)를 기준으로 분할합니다. 단순하지만 토큰 제한을 초과할 위험이 있습니다.
*   **RecursiveCharacterTextSplitter**: 줄바꿈, 문장 구분자 등을 순차적으로 적용하여 재귀적으로 분할합니다. **실무에서 가장 권장되는 방식**입니다.


In [37]:
# [Cell 33]
from langchain.text_splitter import CharacterTextSplitter, RecursiveCharacterTextSplitter

# 실습을 위한 긴 텍스트 (예: 앞서 로드한 PDF의 일부)
text = pages[10].page_content

# CharacterTextSplitter 설정
char_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=500,
    chunk_overlap=50,
    length_function=len
)

# RecursiveCharacterTextSplitter 설정 (권장)
rec_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len
)

print(f" Splitter 설정 완료!")


 Splitter 설정 완료!


[Cell 34]
### 2) 토큰 단위로 분할하기 (tiktoken)
사람이 느끼는 글자 수와 모델이 인식하는 **토큰(Token)** 수는 다릅니다. 토큰 제한을 정확히 지키기 위해 `tiktoken` 라이브러리를 사용합니다.


In [25]:
# [Cell 35]
import tiktoken

tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    tokens = tokenizer.encode(text)
    return len(tokens)

# 글자 수 vs 토큰 수 비교
sample_text = "안녕하세요, LangChain으로 RAG를 만들어봅시다!"
print(f"글자 수: {len(sample_text)}")
print(f"토큰 수: {tiktoken_len(sample_text)}")


글자 수: 31
토큰 수: 21


In [38]:
# [Cell 36]
# 토큰 기준으로 RecursiveCharacterTextSplitter 적용
token_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=tiktoken_len # 글자 수 대신 토큰 수 함수 사용
)

docs = token_splitter.split_documents(pages)

print(f"원본 페이지 수: {len(pages)}")
print(f"분할된 Chunk 수: {len(docs)}")
print(f"\n--- 첫 번째 Chunk 내용 ---")
print(docs[0].page_content[:300])


원본 페이지 수: 182
분할된 Chunk 수: 182

--- 첫 번째 Chunk 내용 ---
DEMIAN 
• 
Downloaded from https://www.holybooks.com


[Cell 37]
## Step 8: Text Embedding 사용해보기

텍스트를 **벡터(숫자 배열)** 로 변환.


In [27]:
# [Cell 41]
from langchain_openai import OpenAIEmbeddings

# OpenAI 임베딩 모델 선언
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

# 테스트: 3개 문장을 벡터로 변환
embeddings = embedding_model.embed_documents([
    "This is red apple.",
    "This is yellow banana.",
    "This is green lime.",
])

print(f"벡터 차원 수: {len(embeddings[0])}")
print(f"벡터 일부: {embeddings[0][:5]}...")


벡터 차원 수: 1536
벡터 일부: [0.003387451171875, -0.0017766952514648438, 0.0007710456848144531, 0.034271240234375, -0.0224151611328125]...


[Cell 38]
### 유사도 계산해보기

코사인 유사도로 벡터 간의 의미적 유사성을 비교합니다.  
"red fruit" 쿼리와 각 과일의 유사도를 확인해보세요!


In [39]:
# [Cell 42]
import numpy as np
from numpy import dot
from numpy.linalg import norm

def cos_sim(A, B):
    return dot(A, B) / (norm(A) * norm(B))

# 새로운 쿼리: "this is red fruit"
query = ["this is red fruit"]
e_query = embedding_model.embed_documents(query)

print(" red apple  유사도:", round(cos_sim(embeddings[0], e_query[0]), 4))
print(" yellow banana 유사도:", round(cos_sim(embeddings[1], e_query[0]), 4))
print(" green lime  유사도:", round(cos_sim(embeddings[2], e_query[0]), 4))

print("\n→ 빨간 사과가 '빨간 과일'과 가장 유사합니다!")


 red apple  유사도: 0.7479
 yellow banana 유사도: 0.4899
 green lime  유사도: 0.4084

→ 빨간 사과가 '빨간 과일'과 가장 유사합니다!


[Cell 40]
## Step 9: VectorStore 사용해보기

Embedding된 벡터를 **저장하고 검색**하는 저장소입니다.  
여기서는 로컬에서 바로 사용할 수 있는 **ChromaDB**를 사용합니다.

이 단계에서 Demian.pdf의 모든 Chunk를 벡터로 변환해서 저장합니다!


In [40]:
# [Cell 44]
from langchain.vectorstores import Chroma

# Chunk들을 임베딩하여 ChromaDB에 저장
db = Chroma.from_documents(docs, embedding_model)

print(f" {len(docs)}개 Chunk가 VectorStore에 저장되었습니다!")


 182개 Chunk가 VectorStore에 저장되었습니다!


In [42]:
# [Cell 45]
# 유사도 검색 테스트
query = "how Demian look like?"
results = db.similarity_search(query)

print(" 검색 결과 (가장 유사한 Chunk):")# 아직 llm을 거치치않음 유사한 chunk 를 찾아 가져오기
print("=" * 60)
print(results[0].page_content)


 검색 결과 (가장 유사한 Chunk):
DEMIAN 
with a feeling of nausea, I noticed Demian's expression. 
He had not thrust himself to the front but stood right 
at the back, looking elc!gant and at ease as usual. His 
glance seemed directed at the horse's head, and again it 
. showed that deep, quiet, almost fanatical yet passionate 
absorption. I could not help staring at him for some 
moments and it was then that I felt aware of a very 
uncanny sensation in my remote consciousness. I saw 
Demian's face and remarked that it was not a boy's face 
but a man's and then I saw, or rather became aware, that 
it was not really the face of a man either; it had some­
thing different about it, almost a feminine element. And 
for the time being his face seemed neither masculine 
nor childish, neither old nor young but a hundred years 
old, almost timeless and bearing the mark of other 
periods of history than our own. Animals might look 
thus, trees or stars. I did not know then, of course, I 
did not feel exac

[Cell 43]
## Step 10: Retriever + QA — RAG 완성!

드디어 모든 조각을 합칩니다!

**RetrievalQA Chain = Retriever + LLM**
1. 사용자 질문 → 벡터 변환
2. VectorStore에서 유사한 Chunk 검색 (Retriever)
3. 검색된 Chunk를 LLM에 전달
4. LLM이 근거 기반 답변 생성

### 주요 파라미터 설명:
- `chain_type="stuff"`: 검색된 문서를 그대로 프롬프트에 삽입
- `search_type="mmr"`: 유사도 + 다양성을 함께 고려하는 검색 방식
- `k=3`: 최종 LLM에 전달할 문서 수
- `fetch_k=10`: 후보로 가져올 문서 수 (k보다 크게 설정)
- `return_source_documents=True`: 출처 문서도 함께 반환


In [44]:
# [Cell 46]
from langchain.chains import RetrievalQA

# RAG QA 체인 구성
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=db.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 3, "fetch_k": 10}
    ),
    return_source_documents=True
)

print(" RAG QA 체인 구성 완료!")


 RAG QA 체인 구성 완료!


In [45]:
# [Cell 47]
#  RAG로 질의응답! llm을 거친 대답 chain flow 를 거침
query = "how demian looks like"
result = qa.invoke(query)#실행 명령

print(" RAG 답변:")
print("=" * 60)
print(result["result"])


 RAG 답변:
Demian's appearance is described as having a face that is not quite boyish or manly, but rather a blend of both with a touch of femininity. His face is said to be different, almost timeless, and bearing the mark of other periods of history. The narrator finds him attractive yet repelling, and sees him as different from the rest, almost like an animal, spirit, or image.


In [46]:
# [Cell 48]
#  출처 문서 확인 (RAG의 핵심 장점!)
print("\n 참고한 출처 문서들:")
print("=" * 60)
for i, doc in enumerate(result["source_documents"]):
    page_num = doc.metadata.get("page", "?")
    print(f"\n--- 출처 {i+1} (페이지 {page_num}) ---")
    print(doc.page_content[:200] + "...")



 참고한 출처 문서들:

--- 출처 1 (페이지 53) ---
DEMIAN 
with a feeling of nausea, I noticed Demian's expression. 
He had not thrust himself to the front but stood right 
at the back, looking elc!gant and at ease as usual. His 
glance seemed directe...

--- 출처 2 (페이지 89) ---
DEMIAN 
Why had it only just dawned on me I It wu Demian'• 
face. 
Later I often comP9Xed the face on the paper with 
Demian's features as l remembered them. They were 
certainly, though similar, not ...

--- 출처 3 (페이지 127) ---
DEMI.\_N 
intense fervour. Then my dream returned at once-our 
gateway and the coat-of-arms, my mother a_!ld the strange 
woman whose features I saw with such a preternatural 
darity that I was able t...


[Cell 47]
##  RAG vs 일반 LLM 비교

같은 질문을 RAG 없이 LLM에게 직접 물어보면 어떻게 다른지 비교해봅시다!


In [47]:
# [Cell 49]
# RAG 없이 LLM에게 직접 질문
llm_only = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.0)
direct_response = llm_only.invoke("how demian looks like")

print(" 일반 LLM 답변 (RAG 없음):")
print("=" * 60)
print(direct_response.content)

print("\n\n RAG 답변 (문서 기반):")
print("=" * 60)
print(result["result"])

print("\n\n 차이점을 관찰해보세요!")
print("- RAG 답변은 실제 원문에 기반한 구체적인 묘사를 포함합니다")
print("- 일반 LLM 답변은 일반적인 지식에 기반한 답변입니다")
print("- RAG는 출처를 제시할 수 있습니다!")


 일반 LLM 답변 (RAG 없음):
Demian is described as having dark hair and piercing blue eyes. He is tall and lean, with a confident and mysterious aura about him. He often wears dark, stylish clothing and carries himself with a sense of quiet intensity. Overall, Demian is a striking and enigmatic figure.


 RAG 답변 (문서 기반):
Demian's appearance is described as having a face that is not quite boyish or manly, but rather a blend of both with a touch of femininity. His face is said to be different, almost timeless, and bearing the mark of other periods of history. The narrator finds him attractive yet repelling, and sees him as different from the rest, almost like an animal, spirit, or image.


 차이점을 관찰해보세요!
- RAG 답변은 실제 원문에 기반한 구체적인 묘사를 포함합니다
- 일반 LLM 답변은 일반적인 지식에 기반한 답변입니다
- RAG는 출처를 제시할 수 있습니다!


[Cell 49]
##  직접 해보기!

아래 셀에서 `query` 변수를 바꿔가며 다양한 질문을 해보세요.  
Demian 소설에 관한 어떤 질문이든 가능합니다!


In [54]:
# [Cell 50]
# 여기에 원하는 질문을 입력하세요!
query = "What does Demian like?"

result = qa.invoke(query)

print(f" 질문: {query}")
print("=" * 60)
print(f" 답변:\n{result['result']}")
print("\n 출처:")
for i, doc in enumerate(result["source_documents"]):
    page = doc.metadata.get("page", "?")
    print(f"  - 출처 {i+1}: 페이지 {page}")


 질문: What does Demian like?
 답변:
Demian is described as being different from the rest of the characters in the text. He is portrayed as having a deep, quiet, almost fanatical yet passionate absorption in things. His face is described as having a timeless quality, almost like that of an animal, a spirit, or an image. The text suggests that Demian is enigmatic and intriguing, with an aura of mystery and uniqueness.

 출처:
  - 출처 1: 페이지 33
  - 출처 2: 페이지 53
  - 출처 3: 페이지 89


[Cell 51]
---

##  정리

이 실습에서 구현한 RAG 파이프라인 전체 흐름:

```
PDF 문서
  ↓ [Document Loader]
Document 객체 (text + metadata)
  ↓ [Text Splitter]
Chunks (작은 텍스트 조각들)
  ↓ [Embedding Model]
Vectors (숫자 벡터)
  ↓ [VectorStore - ChromaDB]
벡터 데이터베이스
  ↓ [Retriever - MMR 검색]
관련 Chunks 검색
  ↓ [LLM - GPT-3.5]
근거 기반 답변 생성!
```

### 수업 핵심 포인트:
1. **Fine-tuning vs RAG**: Fine-tuning은 "기억을 늘리는 것", RAG는 "검색을 잘하게 하는 것"
2. **RAG의 장점**: 재학습 없이 지식 갱신, 출처 제시 가능, 비용 절감
3. **LangChain**: RAG 파이프라인을 모듈식으로 쉽게 구성하는 프레임워크
4. **RAG의 한계**: 검색 정확도, 맥락 혼선, 할루시네이션 → Advanced RAG, AI Agent로 발전
